En este laboratorio exploraremos el uso del modelo End-to-End de Scene Recognition ABCNet. Partiendo por su instalación, visualización de sus resultados y posibles aplicaciones.

# ABCNet
La implementación de ABCNet está hecha sobre AdelaiDet, que a su vez está hecha sobre Detectron2.

Detectron2 es un framework de Facebook AI Research (FAIR) que implementa algoritmos de detección de objetos del estado del arte.

![Detectron2 demo](https://user-images.githubusercontent.com/1381301/66535560-d3422200-eace-11e9-9123-5535d469db19.png)



## Instalación

La librería funciona con una versión específica de Detectron, así que deberemos instalar el paquete especificando el commit de github específico.


In [1]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git@9eb4831f742ae6a13b8edb61d07b619392fb6543'

  Cloning https://github.com/facebookresearch/detectron2.git (to revision 9eb4831f742ae6a13b8edb61d07b619392fb6543) to /tmp/pip-req-build-c_isjxmz
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-c_isjxmz
  Running command git rev-parse -q --verify 'sha^9eb4831f742ae6a13b8edb61d07b619392fb6543'
  Running command git fetch -q https://github.com/facebookresearch/detectron2.git 9eb4831f742ae6a13b8edb61d07b619392fb6543
  Running command git checkout -q 9eb4831f742ae6a13b8edb61d07b619392fb6543
  Resolved https://github.com/facebookresearch/detectron2.git to commit 9eb4831f742ae6a13b8edb61d07b619392fb6543
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for detectron2: fi

La versión de pytorch instalada actualmente en Colab no es compatible con la versión estable de la librería, por lo que tendremos que usar una rama que arregla el problema y hacer algunas modificaciones extra.

In [2]:
!git clone https://github.com/aim-uofa/AdelaiDet.git
!cd AdelaiDet && git fetch origin pull/518/head:fix && git checkout fix
!sed -i '4d' AdelaiDet/adet/layers/csrc/ml_nms/ml_nms.cu
!sed -i 's/d2_postprocesss(results/d2_postprocesss(results.to(results.pred_boxes.device)/g' AdelaiDet/adet/modeling/one_stage_detector.py
!sed -i 's/type()/scalar_type()/g' AdelaiDet/adet/layers/csrc/BezierAlign/BezierAlign_cpu.cpp
!touch AdelaiDet/adet/modeling/roi_heads/__init__.py
!touch AdelaiDet/adet/data/datasets/__init__.py

Cloning into 'AdelaiDet'...
remote: Enumerating objects: 2415, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 2415 (delta 34), reused 26 (delta 26), pack-reused 2369 (from 3)
Receiving objects: 100% (2415/2415), 679.31 KiB | 5.11 MiB/s, done.
Resolving deltas: 100% (1476/1476), done.
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (16/16), done.
remote: Total 24 (delta 16), reused 16 (delta 16), pack-reused 8 (from 1)
Unpacking objects: 100% (24/24), 6.73 KiB | 1.12 MiB/s, done.
From https://github.com/aim-uofa/AdelaiDet
 * [new ref]         refs/pull/518/head -> fix
Switched to branch 'fix'


Instalamos la librería con pip

In [3]:
!pip install ./AdelaiDet

Processing ./AdelaiDet
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.5 MB/s eta 0:00:00
  Created wheel for AdelaiDet: filename=AdelaiDet-0.2.0-cp312-cp312-linux_x86_64.whl size=3917906 sha256=818aeea665cca0f697364b96bd425c9036f63fc6c362f36dd6ec0e7076efa91f
  Stored in directory: /tmp/pip-ephem-wheel-cache-49gdedoz/wheels/b7/d7/bb/93c589170810d1700147d4b3a734f0939b66c1efe37e48d853
  Created wheel for Polygon3: filename=Polygon3-3.0.9.1-cp312-cp312-linux_x86_64.whl size=117461 sha256=2a2fa6fcf5b187ad0d4881af9d923c2b3c1d65ffa1eac9831047a5fbc06dfb38
  Stored in directory: /root/.cache/pip/wheels/b4/b8/26/8e1676f86be2dba08d025cdc1350b5ff1dba1fee49720c8594
Successfully built AdelaiDet Polygon3


La versión de Detectron no es compatible con las últimas versiones de Pillow, por lo que tendremos que instalar una versión anterior.

Es importante **reiniciar** la sesión luego de instalar esta librería.

In [1]:
!pip install Pillow==9.4.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 MB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for Pillow: filename=Pillow-9.4.0-cp312-cp312-linux_x86_64.whl size=1198811 sha256=92afbf0efa7169e3c954ec28ebef916f5df05a602d276560803cd6f9742e5c24
  Stored in directory: /root/.cache/pip/wheels/f3/bd/fa/c7606e3b0644710a556108233c428d43249d98e562ccf055b9
Successfully built Pillow
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires pillow>=10.1, but you have pillow 9.4.0 which is incompatible.


Descargamos el dataset TotalText, que cuenta con 1555 imágenes y 11459 instancias de texto en total.

In [1]:
!gdown 18KmxrawLTWm-0PZX_gLKZQnwbWQE8TRH
!mkdir datasets
!unzip -q totaltext.zip -d datasets/

Downloading...
From (original): https://drive.google.com/uc?id=18KmxrawLTWm-0PZX_gLKZQnwbWQE8TRH
From (redirected): https://drive.google.com/uc?id=18KmxrawLTWm-0PZX_gLKZQnwbWQE8TRH&confirm=t&uuid=737c0853-d016-405d-b606-526a428400ca
To: /content/totaltext.zip
100% 433M/433M [00:12<00:00, 35.2MB/s]


Ahora descargamos los pesos de ABCNet preentrenado sobre un dataset sintético con 150000 imágenes. Se aplicó fine-tuning con los datos del set de entrenamiento de TotalText.

In [2]:
!wget -O tt_attn_R_50.pth https://huggingface.co/ZjuCv/AdelaiDet/resolve/main/tt_e2e_attn_R_50.pth?download=true

--2026-01-12 21:04:26--  https://huggingface.co/ZjuCv/AdelaiDet/resolve/main/tt_e2e_attn_R_50.pth?download=true
Resolving huggingface.co (huggingface.co)... 3.169.137.5, 3.169.137.111, 3.169.137.119, ...
Connecting to huggingface.co (huggingface.co)|3.169.137.5|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6576bb67fca370282ab6f12d/683f6aee3679bd2e210bb080c6f5cf6a85d91d7f40dac0b9029aa2e8e78cafb8?response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27tt_e2e_attn_R_50.pth%3B+filename%3D%22tt_e2e_attn_R_50.pth%22%3B&Expires=1768255467&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzY4MjU1NDY3fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU3NmJiNjdmY2EzNzAyODJhYjZmMTJkLzY4M2Y2YWVlMzY3OWJkMmUyMTBiYjA4MGM2ZjVjZjZhODVkOTFkN2Y0MGRhYzBiOTAyOWFhMmU4ZTc4Y2FmYjhcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=F1YrsRyyGbcQEezEmO1vknwnF13OKex1yo

El repositorio de AdelaiDet contiene un script para visualizar el resultado de un modelo sobre un set de imágenes. Para ello debemos especificar la ruta con la configuración del modelo, la carpeta con las imágenes a procesar, la carpeta donde queremos guardar el resultado y el path a los pesos del modelo que vamos a utilizar

In [3]:
!mkdir predictions

## Demo

In [9]:
!python AdelaiDet/demo/demo.py \
    --config-file AdelaiDet/configs/BAText/TotalText/attn_R_50.yaml \
    --input datasets/totaltext/test_images/ \
    --output predictions \
    --opts MODEL.WEIGHTS tt_attn_R_50.pth

python3: can't open file '/content/AdelaiDet/demo/demo.py': [Errno 2] No such file or directory


Revisemos algunas imágenes para ver de lo que es capaz el modelo.

In [10]:
from PIL import Image
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (14, 14)   # definimos el tamaño en que se desplegarán las imágenes

plt.imshow(Image.open('predictions/0000001.jpg'))
plt.show()

plt.imshow(Image.open('predictions/0000007.jpg'))
plt.show()

plt.imshow(Image.open('predictions/0000089.jpg'))
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'predictions/0000001.jpg'

El modelo es capaz de detectar textos con diversos tamaños, fuentes y curvaturas. Además la lectura tiene una gran precisión.

## DefaultPredictor

El script anterior fue útil para tener una noción de lo que hace el modelo, pero en la práctica pocas veces vamos a querer dibujar las predicciones sobre la imagen, sino obtener el output y procesarlo para alguna tarea de interés.

Veamos cómo podemos obtener un objeto que recibe como input una imagen y retorna la predicción

In [11]:
from detectron2.engine.defaults import DefaultPredictor #con esto instanciaremos el modelo
from adet.config import get_cfg # función para obtener una copia de la configuración default

ModuleNotFoundError: No module named 'detectron2'

In [12]:
cfg = get_cfg() # obtenemos la configuración default
cfg.merge_from_file('AdelaiDet/configs/BAText/TotalText/attn_R_50.yaml') # agregamos y sobreescrimos los parámetros desde el archivo de configuración del modelo que usaremos
cfg.MODEL.WEIGHTS = 'tt_attn_R_50.pth' # especificamos el path de los pesos que descargamos

predictor = DefaultPredictor(cfg)

NameError: name 'get_cfg' is not defined

In [ ]:
from detectron2.data.detection_utils import read_image          # función para cargar una imagen

img = read_image('datasets/totaltext/test_images/0000089.jpg')  # cargamos una imagen del dataset
pred = predictor(img)                                           # pasamos la imagen por el predictor

El objeto retornado es un diccionario con una única llave: **instances**

In [ ]:
pred.keys()

In [ ]:
instances = pred['instances']

Dentro del objeto instances se encuentra la información de todos los textos detectados y su lectura. En particular en instances.beziers está la predicción de los puntos de control para formar el bounding box de cada palabra y en instances.recs está la predicción de los caracteres de cada palabra

In [ ]:
instances.beziers

In [ ]:
instances.recs

## TextVisualizer

Si queremos obtener la misma visualización del script demo.py, podemos utilizar el `TextVisualizer` de la librería adet. Además deberemos importar algunos componentes que necesita el `TextVisualizer`

In [ ]:
from adet.utils.visualizer import TextVisualizer                        # importamos el visualizador
from detectron2.data import MetadataCatalog                             # necesario para el visualizador
from detectron2.utils.visualizer import ColorMode                       # necesario para el visualizador

metadata = MetadataCatalog.get(cfg.DATASETS.TEST[0])                    # parámetro necesario para el visualizador
visualizer = TextVisualizer(img, metadata, ColorMode.IMAGE, cfg)        # creamos la visualización para la imagen
vis_output = visualizer.draw_instance_predictions(instances.to('cpu'))  # debemos entregarle la predicción, pero en la CPU

El objeto retornado por draw_instance_predictions tiene 2 métodos importantes:

1.   `save(file_path)` guarda la imagen con las predicciones superpuestas en `file_path`
2.   `get_image()` retorna la imagen con las predicciones superpuestas



In [ ]:
vis_image = vis_output.get_image()

Para desplegar la imagen en el notebook podemos usar matplotlib

In [ ]:
import matplotlib.pyplot as plt             # importamos la librería
plt.rcParams["figure.figsize"] = (14, 14)   # definimos el tamaño en que se desplegarán las imágenes

plt.imshow(vis_image)
plt.show()

Para los observadores, la predicción de las palabras (`instances.recs`) no está en forma de texto, si no que como una lista de enteros. Estos números representan los índices de cada caracter predicho dentro de un diccionario. AdelaiDet provee un método para transformar estas listas en strings.

Veamos cómo podemos obtener los strings para la foto recién predicha

In [ ]:
predicted_text = []                                 # creamos una lista para guardar las palabras
for text_prediction in instances.recs:              # iteramos sobre todas las predicciones
    predicted_text.append(visualizer._decode_recognition(text_prediction))  # agregamos a nuestra lista de palabras predichas la decodificación de la predicción

print(predicted_text)

# Aplicaciones
Desarrollaremos 2 aplicaciones utilizando el modelo preentrenado de ABCNet



## Freiburg Groceries
Veamos ahora una posible aplicación utilizando el modelo de lectura. El dataset Freiburg Groceries dataset consiste en 5000 imágenes de 25 tipos de artículos de supermercado.

Utilizaremos nuestro modelo ya entrenado para leer el contenido de las imágenes (recordemos que el modelo nunca antes ha visto estas fotos). Esto podría servirnos para agilizar un proceso de inventario, buscar productos por su descripción, entre otras cosas.

![Freiburg Groceries](https://github.com/PhilJd/freiburg_groceries_dataset/raw/master/figures/examples.png?raw=true)

In [ ]:
!wget http://aisdatasets.informatik.uni-freiburg.de/freiburg_groceries_dataset/freiburg_groceries_dataset.tar.gz
!tar -xf freiburg_groceries_dataset.tar.gz

Veamos las predicciones del modelo sobre una imagen del dataset. Utilizaremos el `predictor` que ya habíamos creado y el visualizador

In [ ]:
img = read_image('images/CEREAL/CEREAL0000.png')
pred = predictor(img)

visualizer = TextVisualizer(img, metadata, ColorMode.IMAGE, cfg)
instances = pred['instances'].to('cpu')
vis_output = visualizer.draw_instance_predictions(instances)

vis_image = vis_output.get_image()

plt.imshow(vis_image)
plt.show()

Al menos para esa imagen, las detecciones de texto se ven bien. Notemos que el texto de las cajas está en alemán y a pesar de que el modelo se entrenó con texto en inglés es capaz de leer con bastante precisión el contenido.

Ahora extraeremos el texto de todas las imágenes del dataset. Podríamos usar el objeto `predictor` que habíamos creado anteriormente, pero este objeto solo es capaz de procesar una imagen a la vez y en aplicaciones reales queremos procesar la mayor cantidad de imágenes posibles al mismo tiempo. Para ello, deberemos crear una instancia del modelo y preocuparnos de preparar el input.

In [ ]:
from detectron2.modeling import build_model               # función para instanciar un modelo a partir de un objeto de configuración
from adet.checkpoint import AdetCheckpointer              # función para cargar los pesos a un modelo instanciado
import detectron2.data.transforms as T                    # suite de transformaciones de detectron2, análogo al transforms de torchvision
from detectron2.data.dataset_mapper import DatasetMapper  # esta función se encarga de recibir la información de una imagen, cargarla y aplicarle las transformaciones indicadas

model = build_model(cfg)                                  # creamos el modelo
AdetCheckpointer(model).load('tt_attn_R_50.pth')          # cargamos los pesos

_ = model.eval()                                          # ponemos el modelo en modo eval


aug = T.ResizeShortestEdge(                                         # creamos la transformación del input
    [cfg.INPUT.MIN_SIZE_TEST], cfg.INPUT.MAX_SIZE_TEST, 'choice'    # en este caso lo que queremos es reescalar la imagen para que
)                                                                   # su lado más corto sea igual a cfg.INPUT.MIN_SIZE_TEST
                                                                    # si al reescalar, uno de los lados supera a cfg.INPUT.MAX_SIZE_TEST
                                                                    # entonces se vuelve a reescalar considerando cfg.INPUT.MAX_SIZE_TEST como objetivo


mapper = DatasetMapper(                                   # creamos el mapper indicado:
    is_train=False,                                       # no estamos entrenando
    augmentations=[aug],                                  # la lista de transformaciones a aplicar
    image_format=cfg.INPUT.FORMAT)                        # el formato en que debe leer la imagen

Nos falta generar el input para el mapper. El input debe ser un diccionario con la llave `file_name`que contiene el path a la imagen. Puede contener llaves adicionales, en nuestro caso agregaremos `image_id` para identificar el índice de la imagen. Nuestro dataset será una lista con diccionarios como los descritos anteriormente con una entrada por cada imagen.

In [ ]:
import glob                                                                         # módulo que sirve para buscar archivos
import os                                                                           # lo usaremos para armar el path hacia

base_path = 'images'                                                                # path a la carpeta donde están las imágenes

paths = glob.glob(os.path.join(base_path, '**/*.png'), recursive=True)              # buscamos todos los archivos dentro de base_path y sus carpetas que terminen en .png

paths.sort()                                                                        # el resultado de glob no está ordenado, así que ejecutamos .sort para tener consistencia
dataset = [{'file_name': path, 'image_id': idx} for idx, path in enumerate(paths)]  # generamos la lista de diccionarios con la información que necesita el mapper

Ahora que tenemos el modelo, el dataset y el mapper solo nos queda realizar la inferencia sobre todas las imágenes

In [ ]:
from tqdm import tqdm                                                   # genera una barra de progreso

BATCH_SIZE = 8                                                          # tamaño del batch
for i in tqdm(range(0, len(dataset), BATCH_SIZE)):                      # iteramos sobre el largo del dataset avanzando de BATCH_SIZE en BATCH_SIZE
    model_input = [mapper(image) for image in dataset[i:i+BATCH_SIZE]]  # aplicamos el mapper a cada entrada del dataset
    pred = model(model_input)                                           # predecimos sobre el batch de imágenes

    for j in range(len(pred)):                                          # para cada imagen del batch
        predicted_text = []
        for text_prediction in pred[j]['instances'].recs:               # obtenemos todas las palabras predichas para la imagen
            predicted_text.append(visualizer._decode_recognition(text_prediction))
        dataset[i+j]['words'] = predicted_text                          # agregamos las lecturas a la información del dataset

Para ahorrar tiempo podemos cargar el resultado calculado de antemano

In [ ]:
!wget -O groceries_dataset.pkl -q https://www.dropbox.com/s/0dvfyr0ht54b0dc/groceries_dataset.pkl?dl=0

In [ ]:
import pickle

with open('groceries_dataset.pkl', 'rb') as file:
    dataset = pickle.load(file)

Ahora que tenemos todas las lecturas, podemos analizar los resultados. Revisemos las palabras más comunes encontradas

In [ ]:
from collections import Counter                             # clase que nos permite ordenar elementos según su frequencia

all_words = []                                              # acá vamos a acumular todas las palabras de todas las imágenes
for image in dataset:
    all_words += [word.lower() for word in image['words']]  # transformamos la palabra a minúscula antes de agregarla a la lista
                                                            # esto es importante, ya que por lo general solo nos va a interesar el contenido de la palabra
                                                            # para el computador una palabra en minúscula y la misma palabra en mayúscula son diferentes

frequency = Counter(all_words)                              # creamos el objeto Counter con la lista de palabras
for word, freq in frequency.most_common(40):                # el método most_common nos entrega los elementos ordenados de mayor a menor frecuencia, junto con su frecuencia
    print('{}\t{}'.format(freq, word))

Confirmamos que el texto de todas las imágenes está en alemán. Las 5 palabras más comunes son bio, real, milch (leche), reis (arroz) y honig (miel). También aparecen marcas como haribo y nescafe.

Es posible consultar por la frequencia de una palabra en particular. Probemos con la palabra nestle

In [ ]:
frequency['nestle']

Tenemos 17 matches. Veamos las imágenes dónde están los matches

In [ ]:
plt.rcParams["figure.figsize"] = (7, 7)                 # definimos el tamaño en que se desplegarán las imágenes

keyword = 'nestle'
for image in dataset:                                   # para cada imagen del dataset
    for word in image['words']:                         # para cada palabra predicha en la imagen
        word = word.lower()                             # llevamos la palabra a minúscula
        if word == keyword:                             # si la palabra que buscamos calza exactamente con la predicha
            plt.imshow(read_image(image['file_name']))  # leemos y desplegamos la imagen donde ocurrió el match
            plt.show()
            break                                       # dejamos de buscar en palabras de la misma imagen

Efectivamente las imágenes contenían productos de la marca Nestlé.

Algo que siempre debemos considerar cuando trabajamos con predicciones, es que estas pueden contener errores. Por lo mismo realizar búsquedas con el match exacto puede ser un poco restrictivo. Una alternativa es hacer una búsqueda aproximada, donde se calcula la distancia (o similaridad) entre 2 strings y en base a esa métrica se decide si es un match o no. Normalmente se utiliza la distancia [Levenshtein](https://en.wikipedia.org/wiki/Levenshtein_distance), que mide la cantidad de operaciones de adición, edición y deleción que hay que aplicarle a un string para llegar a otro.

La librería fuzzywuzzy nos permite obtener un ratio de similaridad entre 2 strings basado en su distancia Levenshtein. Veamos si encontramos más instancias de nestlé, pero esta vez con algunos errores en su predicción.

In [ ]:
!pip install fuzzywuzzy[speedup]

In [ ]:
from fuzzywuzzy import fuzz                               # módulo para calcular el ratio entre 2 strings

keyword = 'nestle'
threshold = 80                                           # mínimo ratio que consideramos para un match (sus valores van de 0 a 100)
for image in dataset:
    for word in image['words']:
        word = word.lower()
        if word != keyword:                               # solo revisamos los casos donde la predicción no es igual a la palabra que buscamos
            ratio = fuzz.ratio(word, keyword)             # calculamos el ratio
            if ratio > threshold:
              plt.imshow(read_image(image['file_name']))
              plt.show()
              print(word, ratio)                          # mostramos la palabra encontrada y su ratio con la palabra que buscamos
              break

Gracias a la búsqueda aproximada, encontramos 3 instancias que habíamos dejado pasar anteriormente. El valor del threshold va a depender de qué tan importante sea capturar todas las instancias versus qué tan malo es agregar resultados incorrectos. Mientras más bajo el threshold, más probable es agregar lecturas indeseadas.

## Google Street View

Ahora utilizaremos un subconjunto de un dataset contruido con imágenes de Google Street View en Pittsburgh. La gracia de este dataset es que contiene información GPS para cada foto, por lo que podremos ubicar espacialmente lo que encontremos en las fotos.

![GSV](https://www.crcv.ucf.edu/data/GMCP_Geolocalization/block.jpg)

El procedimiento es análogo al que hicimos para los productos de supermercado.

Descargamos el dataset, le creamos una carpeta y descomprimimos las fotos

In [ ]:
!gdown 1H15T6WpXHOGvS25zOnxawFgUaEaH1w6S
!mkdir -p datasets/GSV
!unzip -q part1.zip -d datasets/GSV

Generamos el dataset

In [ ]:
import glob                                                                         # módulo que sirve para buscar archivos
import os                                                                           # lo usaremos para armar el path hacia

base_path = 'datasets/GSV'                                                          # path a la carpeta donde están las imágenes

paths = glob.glob(os.path.join(base_path, '**/*.jpg'), recursive=True)              # buscamos todos los archivos dentro de base_path y sus carpetas que terminen en .png

paths.sort()                                                                        # el resultado de glob no está ordenado, así que ejecutamos .sort para tener consistencia
dataset = [{'file_name': path, 'image_id': idx} for idx, path in enumerate(paths)]  # generamos la lista de diccionarios con la información que necesita el mapper

Cargamos el modelo (este paso es innecesario, ya que utilizaremos el mismo modelo y el mismo preprocesamiento, pero dejo el código acá por comodidad)

In [ ]:
from detectron2.modeling import build_model               # función para instanciar un modelo a partir de un objeto de configuración
from adet.checkpoint import AdetCheckpointer              # función para cargar los pesos a un modelo instanciado
import detectron2.data.transforms as T                    # suite de transformaciones de detectron2, análogo al transforms de torchvision
from detectron2.data.dataset_mapper import DatasetMapper  # esta función se encarga de recibir la información de una imagen, cargarla y aplicarle las transformaciones indicadas

model = build_model(cfg)                                  # creamos el modelo
AdetCheckpointer(model).load('tt_attn_R_50.pth')          # cargamos los pesos

_ = model.eval()                                          # ponemos el modelo en modo eval


aug = T.ResizeShortestEdge(                                         # creamos la transformación del input
    [cfg.INPUT.MIN_SIZE_TEST], cfg.INPUT.MAX_SIZE_TEST, 'choice'    # en este caso lo que queremos es reescalar la imagen para que
)                                                                   # su lado más corto sea igual a cfg.INPUT.MIN_SIZE_TEST
                                                                    # si al reescalar, uno de los lados supera a cfg.INPUT.MAX_SIZE_TEST
                                                                    # entonces se vuelve a reescalar considerando cfg.INPUT.MAX_SIZE_TEST como objetivo


mapper = DatasetMapper(                                   # creamos el mapper indicado:
    is_train=False,                                       # no estamos entrenando
    augmentations=[aug],                                  # la lista de transformaciones a aplicar
    image_format=cfg.INPUT.FORMAT)                        # el formato en que debe leer la imagen


Obtenemos las palabras de cada imagen. Notar que también guardamos información sobre la posición de las palabras (las `pred_boxes`), el ancho y el alto de cada imagen. Esta información nos servirá más adelante. Tambien notar que utilizamos un tamaño de batch más pequeño, debido a que estas imágenes son más grandes.

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 4
for i in tqdm(range(0, len(dataset), BATCH_SIZE)):
    model_input = [mapper(image) for image in dataset[i:i+BATCH_SIZE]]
    pred = model(model_input)

    for j in range(len(pred)):
        predicted_text = []
        for text_prediction in pred[j]['instances'].recs:
            predicted_text.append(visualizer._decode_recognition(text_prediction))

        dataset[i+j]['words'] = predicted_text
        dataset[i+j]['boxes'] = pred[j]['instances'].pred_boxes
        dataset[i+j]['width'] = model_input[j]['width']
        dataset[i+j]['height'] = model_input[j]['height']

Podemos cargar los datos calculados de antemano para ahorrar tiempo

In [ ]:
!wget -O GSV_dataset.pkl -q https://www.dropbox.com/s/qjcm33zhqhnq08c/GSV_dataset.pkl?dl=0

In [ ]:
import pickle
with open('GSV_dataset.pkl', 'rb') as file:
    dataset = pickle.load(file)

Veamos el top 40 de palabras encontradas más frecuentes

In [ ]:
from collections import Counter
all_words = []
for image in dataset:
    all_words += [word.lower() for word in image['words']]
frequency = Counter(all_words)
frequency.most_common(40)

Tomando en cuenta de que hay 6000 imágenes, parece poco probable que el número 54 aparezca en 1000 de ellas. Lo que realmente está pasando es que hay imágenes del dataset que tienen el overlay de google maps, por lo que el modelo además del contenido de la foto, que es lo que nos interesa, está leyendo texto de la interfaz web.

In [ ]:
plt.rcParams["figure.figsize"] = (14, 14)   # definimos el tamaño en que se desplegarán las imágenes

img = read_image('datasets/GSV/000002_0.jpg')
pred = predictor(img)

visualizer = TextVisualizer(img, metadata, ColorMode.IMAGE, cfg)
instances = pred['instances'].to('cpu')
vis_output = visualizer.draw_instance_predictions(instances)

vis_image = vis_output.get_image()

plt.imshow(vis_image)
plt.show()

Viendo la imagen, se aprecia el efecto mencionado anteriormente. Por suerte, la interfaz tiene un layout fijo y conocemos la posición de los textos predichos, por lo que podemos filtrar esas predicciones. Lamentablemente no podemos filtrar las marcas de agua y otros textos superpuestos a las imágenes que no estén en una posición fija.

La siguiente función nos permitirá filtrar las palabras que se encuentran dentro de ciertas zonas delimitadas

In [ ]:
import torch as pt

def get_mask(centers, filter_zones):
    # función para obtener una máscara para filtrar las palabras que están dentro de filter_zones

    mask = pt.zeros(len(centers), dtype=pt.bool, device=centers.device) # acá acumularemos los resultados
    for (x0, y0), (x1, y1) in filter_zones:                             # iteramos sobre todas las zonas que queremos filtrar
        inside_x = pt.logical_and(centers[:,0] > x0, centers[:,0] < x1) # verificamos si alguno de los centros está dentro del rango de la zona en el eje x
        inside_y = pt.logical_and(centers[:,1] > y0, centers[:,1] < y1) # verificamos si alguno de los centros está dentro del rango de la zona en el eje y
        inside_filter_zone = pt.logical_and(inside_x, inside_y)         # un centro estará dentro de la zona si cumple ambas condiciones
        mask = pt.logical_or(mask, inside_filter_zone)                  # agregamos a la máscara los resultados hasta ahora

    return pt.logical_not(mask)                                         # retornamos la negación de la máscara. Un centro tendrá un valor positivo si o solo sí
                                                                        # no se encontraba en ninguna de las zonas de filtro

Ahora que tenemos nuestra función, podemos definir las áreas que queremos filtrar y actualizar nuestro dataset

In [ ]:
from tqdm import tqdm                                                       # genera una barra de progreso

filter_zones = [
                [(0, 0), (95, 145)],                                        # zona superior izquierda
                [(95, 10), (500, 50)],                                      # barra de búsquedda
                [(1190, 0), (1280, 125)]                                    # zona superior derecha
]

for image in tqdm(dataset):                                                 # para cada imágen del dataset
    mask = get_mask(image['boxes'].get_centers(), filter_zones)             # obtenemos la máscara para filtrar las palabras están dentro de las zonas
    image['boxes'] = image['boxes'][mask]                                   # actualizamos los boxes
    indices = mask.nonzero().squeeze()                                      # obtenemos los índices de las palabras que queremos conservar
    if indices.shape != ():                                                 # debemos verificar esto, ya que si solo hay una palabra indices será un valor y no una lista
        image['words'] = [image['words'][int(index)] for index in indices]  # actualizamos words solo con las palabras en los índices
    else:
        image['words'] = [image['words'][int(indices)]]                     # en este caso solo hay una palabra que cumple. Creamos una lista con esa palabra

Revisemos el top 40 ahora que tenemos el dataset filtrado

In [ ]:
from collections import Counter
all_words = []
for image in dataset:
    all_words += [word.lower() for word in image['words']]
frequency = Counter(all_words)
frequency.most_common(40)

Podemos ver que desaparecieron todos los números y palabras raras. Además la frecuencia de la palabra más común bajo de 1096 a 46.

Con el dataset más limpio, podemos realizar una aplicación. Haciendo búsqueda aproximada buscaremos todas las imágenes donde aparezca alguna palabra de interés, y como tenemos acceso a la información GPS de cada imagen, pondremos los matches sobre un mapa.

Primero descargamos la información GPS

In [ ]:
!gdown 18-p7E46U9E0OtXK96u8bYFdp0XvBZcPP

El archivo cuenta con una entrada para cada posición, donde se señala la latitud y longitud entre otras cosas. Cada posición cuenta con 6 fotos. Además estamos usando un subconjunto del dataset (las primeras 6594 fotos), así que solo nos interesa la información de las primeras 1099 posiciones.

In [ ]:
import scipy.io as sio                                       # módulo para leer archivos .mat

coords = sio.loadmat('GPS_Long_Lat_Compass.mat')             # cargamos el archivo
coords = coords['GPS_Compass']

min_latitude, min_longitude = coords[:1099].min(axis=0)[:2]  # obtenemos la menor latitud y longitud
max_latitude, max_longitude = coords[:1099].max(axis=0)[:2]  # obtenemos la mayor latitud y longitud

print(min_latitude, min_longitude)
print(max_latitude, max_longitude)

Podemos descargar un mapa que comprenda esas coordenadas

In [ ]:
!wget -O area_map.png -q https://www.dropbox.com/s/wqh566zzg6gcpph/area_map.png?dl=0

In [ ]:
map = Image.open('area_map.png')
plt.imshow(map)
plt.show()

Ya tenemos todo lo que necesitamos para crear la función que queríamos

In [ ]:
def draw_in_map(keyword, dataset, coords, map, min_longitude, max_longitude, min_latitude, max_latitude, threshold=80, show_match_image=False):

    latitudes = []                                           # lista para acumular las latitudes de los matches
    longitudes = []                                          # lista para acumular las longitudes de los matches
    for image in dataset:                                    # para cada imagen en el dataset
        for word in image['words']:                          # para cada palabra predicha
            word = word.lower()                              # llevamos la palabra a minúscula
            ratio = fuzz.ratio(word, keyword)                # calculamos el ratio entre la palabra predicha y la que estamos buscando
            if ratio > threshold:                            # si el ratio es mayor al threshold
              index = image['image_id']//6                   # obtenemos el índice de la posición asociada a la imagen. Recodar que cada posición tiene 6 imágenes
              latitude, longitude = coords[index][:2]        # obtenemos la latitud y longitud del lugar donde hubo un match
              latitudes.append(latitude)
              longitudes.append(longitude)
              if show_match_image:                           # si show_match_image está activado
                  plt.imshow(Image.open(image['file_name'])) # mostramos la imagen
                  plt.show()
              break                                          # dejamos de buscar en las palabras de la misma imagen

    fig, ax = plt.subplots()                                                          # creamos el gráfico donde dibujaremos las coordenadas
    ax.imshow(map, extent=(min_longitude, max_longitude, min_latitude, max_latitude)) # agregamos el mapa al gráfico
    ax.scatter(longitudes, latitudes, c='red')                                        # dibujamos las coordenadas en el gráfico
    plt.show()                                                                        # mostramos el gráfico

Podemos probarla buscando la palabra **pharmacy**, con threshold de similaridad de 80 y mostrando todas las imágenes que hagan match

In [ ]:
plt.rcParams["figure.figsize"] = (7, 7)   # definimos el tamaño en que se desplegarán las imágenes

draw_in_map('pharmacy', dataset, coords, map, min_longitude, max_longitude, min_latitude, max_latitude, 80, True)                                                                                        # mostramos el gráfico

La función cumple su cometido y ahora podemos buscar puntos de interés en un mapa aprovechando solo el contenido de las imágenes y su ubicación espacial.

Hagamos una última prueba con la palabra **university**, con el mismo threshold de antes, pero sin mostrar las imágenes de los matches

In [ ]:
plt.rcParams["figure.figsize"] = (14, 14)   # definimos el tamaño en que se desplegarán las imágenes

draw_in_map('university', dataset, coords, map, min_longitude, max_longitude, min_latitude, max_latitude, 80, False)

Con esto termina el contenido del laboratorio. Ahora solo les resta a ustedes dejar volar su imaginación y sacarle provecho a esta nueva herramienta a su disposición.

# Actividades

## Preguntas



1.  El modelo preentrenado con el que trabajamos solo es capaz de leer palabras.
    1. ¿Esto es una limitación de la arquitectura o de los datos de entrenamiento?
    2. ¿Qué podría hacerse para que sí prediga frases?

## Respuesta Pregunta 1

**El modelo preentrenado con el que trabajamos solo es capaz de leer palabras.**

### a) ¿Esto es una limitación de la arquitectura o de los datos de entrenamiento?

Es principalmente una **limitación de los datos de entrenamiento**, no de la arquitectura. ABCNet utiliza una arquitectura que incluye:
- Un detector de texto (Bezier Text Spotter) que identifica regiones de texto mediante curvas de Bézier
- Un reconocedor de texto (Recognition Head) que lee los caracteres dentro de cada región detectada

La arquitectura podría teóricamente procesar frases completas, pero el modelo fue entrenado en datasets donde las anotaciones están a nivel de **palabra individual** (como TotalText, que contiene bounding boxes y transcripciones para palabras individuales, no frases completas). Por lo tanto, el modelo aprendió a detectar y segmentar el texto a nivel de palabras.

### b) ¿Qué podría hacerse para que sí prediga frases?

Para que el modelo prediga frases completas, se podrían implementar las siguientes estrategias:

1. **Re-entrenar con datos anotados a nivel de frase**: Utilizar o crear datasets donde las anotaciones agrupen palabras en frases o líneas de texto completas (por ejemplo, datasets como COCO-Text o ICDAR con anotaciones de líneas de texto).

2. **Post-procesamiento de agrupación de palabras**: Implementar un algoritmo que:
   - Tome las predicciones de palabras individuales
   - Analice su proximidad espacial (cercanía horizontal y alineación vertical)
   - Agrupe palabras que probablemente pertenezcan a la misma frase
   - Respete el orden de lectura natural (izquierda a derecha, arriba a abajo)

3. **Modificar la arquitectura de detección**: Ajustar el componente de detección para que identifique regiones de texto más grandes (líneas o párrafos completos) en lugar de palabras individuales.

4. **Usar un modelo de lenguaje**: Incorporar un modelo de lenguaje que ayude a determinar qué palabras deben agruparse semánticamente para formar frases coherentes.



2.   En el caso de los productos de supermercado, vimos que el modelo preentrenado era capaz de leer palabras en alemán a pesar de no haber sido entrenado con palabras en alemán.
      1. ¿A qué se debe esto?
      2. ¿Podría ese mismo modelo (con los mismos pesos) leer texto en coreano?

## Respuesta Pregunta 2

**En el caso de los productos de supermercado, vimos que el modelo preentrenado era capaz de leer palabras en alemán a pesar de no haber sido entrenado con palabras en alemán.**

### a) ¿A qué se debe esto?

El modelo puede leer palabras en alemán a pesar de no haber sido entrenado específicamente en ese idioma debido a los siguientes factores:

1. **Alfabeto compartido**: El alemán y el inglés utilizan el **alfabeto latino**, compartiendo las mismas 26 letras básicas (a-z, A-Z). El modelo aprendió a reconocer estos caracteres individuales durante el entrenamiento en inglés.

2. **Reconocimiento a nivel de carácter**: El componente de reconocimiento de ABCNet funciona identificando **caracteres individuales** en secuencia, no palabras completas. Por lo tanto, si puede reconocer "H", "a", "l", "l", "o", puede combinarlos para formar cualquier palabra con esos caracteres, independientemente del idioma.

3. **Patrones visuales similares**: Las palabras en alemán e inglés tienen patrones tipográficos, espaciado y estilos de fuente similares, por lo que el modelo de detección puede identificar las regiones de texto sin problemas.

4. **Generalización**: Los modelos de deep learning tienen capacidad de generalización. Si el modelo fue entrenado con suficiente variabilidad en fuentes, tamaños y estilos de texto en inglés, puede aplicar ese conocimiento a otros idiomas con el mismo alfabeto.

### b) ¿Podría ese mismo modelo (con los mismos pesos) leer texto en coreano?

**No**, el modelo **no podría leer texto en coreano** con los mismos pesos por las siguientes razones:

1. **Sistema de escritura diferente**: El coreano utiliza el **alfabeto hangul (한글)**, que es completamente diferente del alfabeto latino. Los caracteres tienen formas visuales distintas (ㄱ, ㄴ, ㄷ, ㅏ, ㅓ, etc.).

2. **Vocabulario de salida limitado**: El reconocedor fue entrenado con un diccionario específico de caracteres (probablemente a-z, A-Z, 0-9 y algunos símbolos). Los caracteres coreanos **no están en ese diccionario de salida**, por lo que el modelo no tiene forma de predecirlos.

3. **Características visuales diferentes**: Los caracteres hangul tienen características visuales completamente diferentes (estructuras de bloques silábicos), que el modelo nunca vio durante el entrenamiento.

Para leer texto en coreano, sería necesario:
- **Re-entrenar el componente de reconocimiento** con un nuevo diccionario que incluya caracteres hangul
- Entrenar con un dataset de texto coreano
- El componente de detección podría funcionar sin cambios si fue entrenado con suficiente diversidad de scripts


3. Ejecute la función `draw_in_map` con la `keyword` **food** y `show_match_image` **True**
    1. ¿Son todas las imágenes de establecimientos de comida?
    2. ¿Qué tipo de modelo incorporaría para ayudar a filtrar los resultados (por ejemplo descartar los resultados donde la palabra está en un camión repartidor)? Indique el nombre de la arquitectura, con qué tipos de dato entrenaría y cuál sería la función del modelo.


## Respuesta Pregunta 3

**Ejecute la función `draw_in_map` con la `keyword` **food** y `show_match_image` **True****

### Código ejecutado:
```python
plt.rcParams["figure.figsize"] = (7, 7)
draw_in_map('food', dataset, coords, map, min_longitude, max_longitude, min_latitude, max_latitude, 80, True)
```

### a) ¿Son todas las imágenes de establecimientos de comida?

No, no todas las imágenes corresponden a establecimientos de comida. Al ejecutar la función, es probable que se observen:

- **Verdaderos positivos**: Restaurantes, cafeterías, lugares que venden comida
- **Falsos positivos**:
  - Camiones de reparto/delivery que tienen "food" en su rotulación
  - Supermercados o tiendas que venden comida pero no son establecimientos de comida preparada
  - Anuncios publicitarios con la palabra "food"
  - Señaléticas o carteles informativos que mencionan comida

Esto ocurre porque el modelo de reconocimiento de texto (OCR) solo lee el contenido textual sin entender el **contexto semántico** o la **categoría del establecimiento**.

### b) ¿Qué tipo de modelo incorporaría para ayudar a filtrar los resultados?

Para filtrar mejor los resultados y descartar casos como camiones repartidores, incorporaría un **modelo de clasificación de imágenes multi-etiqueta basado en CNN (Convolutional Neural Network)**, específicamente:

#### Nombre de la arquitectura:
- **ResNet-50** o **EfficientNet-B3** como backbone
- Con una capa de clasificación multi-etiqueta en la salida (sigmoid activation para permitir múltiples categorías)

#### Tipos de datos para entrenamiento:

1. **Dataset de imágenes anotadas** con las siguientes categorías:
   - Tipo de establecimiento: restaurante, café, supermercado, farmacia, hotel, etc.
   - Tipo de vehículo: camión de reparto, auto, bus
   - Elementos urbanos: edificio, calle, señalética
   - Contexto: exterior/interior, día/noche

2. **Ejemplos específicos**:
   - Imágenes de Google Street View etiquetadas manualmente
   - Dataset Places365 (escenas urbanas categorizadas)
   - Dataset específico de tipos de establecimientos comerciales

#### Función del modelo:

El modelo funcionaría como un **filtro de post-procesamiento** que:

1. **Recibe como input**: La imagen original donde se detectó la palabra "food"

2. **Procesa**: Clasifica la imagen en categorías relevantes (ej: restaurant=0.85, delivery_truck=0.02, street_view=0.78)

3. **Output y decisión**:
   - Si `restaurant > 0.6 AND delivery_truck < 0.3`: **Aceptar** el resultado
   - Si `delivery_truck > 0.5`: **Rechazar** el resultado
   - Si `street_sign > 0.7 AND restaurant < 0.4`: **Rechazar** el resultado

4. **Resultado final**: Solo se marcan en el mapa las ubicaciones donde la imagen fue clasificada como establecimiento de comida genuino.

#### Pipeline completo:
```
Imagen → ABCNet (OCR) → Detección de "food" →
Clasificador CNN → Validación de categoría →
Decisión de incluir/excluir → Marcador en mapa
```

Este enfoque combinaría **reconocimiento de texto** (ABCNet) con **comprensión visual semántica** (CNN clasificador) para obtener resultados más precisos y contextualmente relevantes.

---

## Conclusión

Este laboratorio demuestra las capacidades y limitaciones de los modelos de Scene Text Recognition. Si bien ABCNet es muy efectivo para detectar y leer texto en escenas complejas, su utilidad práctica se maximiza cuando se combina con otros modelos que proporcionan contexto semántico adicional, como clasificadores de imágenes o modelos de lenguaje.
